In [1]:
print("Olist SQL Notebook Kernel Connected Successfully!")

Olist SQL Notebook Kernel Connected Successfully!


In [2]:
# ============================================================
# SQLITE DATABASE SETUP
# ============================================================

import sqlite3
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = PROJECT_ROOT / "Data" / "Processed"
SQL_PATH = PROJECT_ROOT / "SQL"

# Create SQL folder if it doesn't exist
SQL_PATH.mkdir(parents=True, exist_ok=True)

# SQLite database path
DB_PATH = SQL_PATH / "olist_business.db"

# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)

# Processed CSV files
tables = {
    "orders_analytics": "orders_analytics.csv",
    "order_items_analytics": "order_items_analytics.csv",
    "customer_summary": "customer_summary.csv",
    "category_summary": "category_summary.csv",
    "seller_summary": "seller_summary.csv",
    "monthly_sales": "monthly_sales.csv",
    "state_sales": "state_sales.csv",
    "payment_summary": "payment_summary.csv",
    "review_summary": "review_summary.csv"
}

# Load CSV files into SQLite
for table_name, file_name in tables.items():

    file_path = PROCESSED_DATA_PATH / file_name

    df = pd.read_csv(file_path)

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    print(
        f"Loaded: {table_name:<25} "
        f"Rows: {len(df):,}"
    )

print("\n" + "=" * 60)
print("SQLITE DATABASE CREATED SUCCESSFULLY")
print("=" * 60)

print(f"\nDatabase location:\n{DB_PATH}")

Loaded: orders_analytics          Rows: 99,441
Loaded: order_items_analytics     Rows: 112,650
Loaded: customer_summary          Rows: 96,096
Loaded: category_summary          Rows: 73
Loaded: seller_summary            Rows: 3,095
Loaded: monthly_sales             Rows: 25
Loaded: state_sales               Rows: 27
Loaded: payment_summary           Rows: 99,440
Loaded: review_summary            Rows: 98,673

SQLITE DATABASE CREATED SUCCESSFULLY

Database location:
c:\Users\Shivam kumar\Desktop\Olist-Ecommerce-Business-Analytics\SQL\olist_business.db


In [3]:
# ============================================================
# VERIFY SQLITE TABLES
# ============================================================

tables_check = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

print("Tables in SQLite database:\n")
print(tables_check.to_string(index=False))

Tables in SQLite database:

                 name
     category_summary
     customer_summary
        monthly_sales
order_items_analytics
     orders_analytics
      payment_summary
       review_summary
       seller_summary
          state_sales


In [4]:
# ============================================================
# SQL QUERY 01 — TOP 10 PRODUCT CATEGORIES
# ============================================================

query = """
SELECT
    product_category_name_english AS category,
    items_sold,
    orders,
    ROUND(product_sales, 2) AS product_sales
FROM category_summary
ORDER BY product_sales DESC
LIMIT 10;
"""

top_categories = pd.read_sql_query(query, conn)

print(top_categories.to_string(index=False))

             category  items_sold  orders  product_sales
        health_beauty        9670    8836     1258681.34
        watches_gifts        5991    5624     1205005.68
       bed_bath_table       11115    9417     1036988.68
       sports_leisure        8641    7720      988048.97
computers_accessories        7827    6689      911954.32
      furniture_decor        8334    6449      729762.49
           cool_stuff        3796    3632      635290.85
           housewares        6964    5884      632248.66
                 auto        4235    3897      592720.11
         garden_tools        4347    3518      485256.46


In [5]:
# ============================================================
# SQL QUERY 02 — MONTHLY SALES PERFORMANCE


query = """
SELECT
    year_month,
    orders,
    ROUND(payment_value, 2) AS payment_value
FROM monthly_sales
ORDER BY year_month;
"""

monthly_sales_sql = pd.read_sql_query(query, conn)

print(monthly_sales_sql.to_string(index=False))

year_month  orders  payment_value
2016-09-01       4         252.24
2016-10-01     324       59090.48
2016-12-01       1          19.62
2017-01-01     800      138488.04
2017-02-01    1780      291908.01
2017-03-01    2682      449863.60
2017-04-01    2404      417788.03
2017-05-01    3700      592918.82
2017-06-01    3245      511276.38
2017-07-01    4026      592382.92
2017-08-01    4331      674396.32
2017-09-01    4285      727762.45
2017-10-01    4631      779677.88
2017-11-01    7544     1194882.80
2017-12-01    5673      878401.48
2018-01-01    7269     1115004.18
2018-02-01    6728      992463.34
2018-03-01    7211     1159652.12
2018-04-01    6939     1160785.48
2018-05-01    6873     1153982.15
2018-06-01    6167     1023880.50
2018-07-01    6292     1066540.75
2018-08-01    6512     1022425.32
2018-09-01      16        4439.54
2018-10-01       4         589.67


In [6]:
# ============================================================
# SQL QUERY 03 — TOP 10 STATES BY SALES
# ============================================================

query = """
SELECT
    customer_state AS state,
    customers,
    orders,
    ROUND(payment_value, 2) AS payment_value,
    sales_share_pct
FROM state_sales
ORDER BY payment_value DESC
LIMIT 10;
"""

top_states_sql = pd.read_sql_query(query, conn)

print(top_states_sql.to_string(index=False))

state  customers  orders  payment_value  sales_share_pct
   SP      40302   41746     5998226.96            37.47
   RJ      12384   12852     2144379.69            13.39
   MG      11259   11635     1872257.26            11.70
   RS       5277    5466      890898.54             5.57
   PR       4882    5045      811156.38             5.07
   SC       3534    3637      623086.43             3.89
   BA       3277    3380      616645.82             3.85
   DF       2075    2140      355141.08             2.22
   GO       1952    2020      350092.31             2.19
   ES       1964    2033      325967.55             2.04


In [7]:
# ============================================================
# SQL QUERY 04 — CUSTOMER RETENTION
# ============================================================

query = """
SELECT
    customer_type,
    COUNT(*) AS customers,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customer_summary),
        2
    ) AS customer_percentage
FROM customer_summary
GROUP BY customer_type
ORDER BY customers DESC;
"""

customer_retention_sql = pd.read_sql_query(query, conn)

print(customer_retention_sql.to_string(index=False))

customer_type  customers  customer_percentage
     One-Time      93099                96.88
       Repeat       2997                 3.12


In [8]:
payments_df = pd.read_csv(
    PROCESSED_DATA_PATH.parent / "Raw" / "olist_order_payments_dataset.csv"
)

payments_df.to_sql(
    "order_payments",
    conn,
    if_exists="replace",
    index=False
)

print("Payment table loaded successfully")
print(f"Rows: {len(payments_df):,}")

Payment table loaded successfully
Rows: 103,886


In [9]:
query = """
SELECT
    payment_type,
    COUNT(*) AS payment_records,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS avg_payment_value,
    ROUND(
        SUM(payment_value) * 100.0 /
        (SELECT SUM(payment_value) FROM order_payments),
        2
    ) AS share_pct
FROM order_payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;
"""

payment_analysis_sql = pd.read_sql_query(query, conn)

print(payment_analysis_sql.to_string(index=False))

payment_type  payment_records  total_payment_value  avg_payment_value  share_pct
 credit_card            76795          12542084.19             163.32      78.34
      boleto            19784           2869361.27             145.03      17.92
     voucher             5775            379436.87              65.70       2.37
  debit_card             1529            217989.79             142.57       1.36
 not_defined                3                 0.00               0.00       0.00


In [10]:
query = """
SELECT
    seller_id,
    items_sold,
    orders,
    ROUND(product_sales, 2) AS product_sales,
    ROUND(freight_revenue, 2) AS freight_revenue,
    ROUND(total_value, 2) AS total_value
FROM seller_summary
ORDER BY product_sales DESC
LIMIT 10;
"""

top_sellers_sql = pd.read_sql_query(query, conn)

print(top_sellers_sql.to_string(index=False))

                       seller_id  items_sold  orders  product_sales  freight_revenue  total_value
4869f7a5dfa277a7dca6462dcf3b52b2        1156    1132      229472.63         20168.07    249640.70
53243585a1d6dc2643021fd1853d8905         410     358      222776.05         13080.63    235856.68
4a3ca9315b744ce9f8e9374361493884        1987    1806      200472.92         35067.04    235539.96
fa1c13f2614d7b5c4749cbc52fecda94         586     585      194042.03         10042.70    204084.73
7c67e1448b00f6e969d365cea6b010ab        1364     982      187923.89         51612.55    239536.44
7e93a43ef30c4f03f38b393420bc753a         340     336      176431.87          6322.18    182754.05
da8622b14eb17ae2831f4ac5b9dab84a        1551    1314      160236.57         24955.75    185192.32
7a67c85e85bb2ce8582c35f2203ad736        1171    1160      141745.53         20902.85    162648.38
1025f0e2d44d7041d6cf58b6550e0bfa        1428     915      138968.55         33892.14    172860.69
955fee9216a65b617aa5

In [11]:
query = """
SELECT
    delivery_performance,
    COUNT(*) AS orders,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*)
         FROM orders_analytics
         WHERE order_status = 'delivered'),
        2
    ) AS percentage,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delay_days
FROM orders_analytics
WHERE order_status = 'delivered'
GROUP BY delivery_performance
ORDER BY orders DESC;
"""

delivery_analysis_sql = pd.read_sql_query(query, conn)

print(delivery_analysis_sql.to_string(index=False))

delivery_performance  orders  percentage  avg_delivery_days  avg_delay_days
               Early   88644       91.88              10.88          -13.71
                Late    6534        6.77              33.95           10.62
             On Time    1292        1.34              19.22            0.00
       Not Delivered       8        0.01                NaN             NaN


In [12]:
query = """
SELECT
    o.delivery_performance,
    COUNT(r.order_id) AS reviewed_orders,
    ROUND(AVG(r.average_review_score), 2) AS avg_review_score
FROM orders_analytics o
JOIN review_summary r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY o.delivery_performance
ORDER BY avg_review_score DESC;
"""

delivery_review_sql = pd.read_sql_query(query, conn)

print(delivery_review_sql.to_string(index=False))

delivery_performance  reviewed_orders  avg_review_score
       Not Delivered                8              4.50
               Early            88163              4.29
             On Time             1280              4.04
                Late             6381              2.27


In [13]:
query = """
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    ROUND(
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        2
    ) AS delivery_completion_rate,
    ROUND(SUM(total_payment_value), 2) AS total_payment_value,
    ROUND(AVG(total_payment_value), 2) AS average_order_value,
    ROUND(AVG(average_review_score), 2) AS average_review_score
FROM orders_analytics;
"""

final_kpis_sql = pd.read_sql_query(query, conn)

print(final_kpis_sql.to_string(index=False))

 total_orders  delivered_orders  delivery_completion_rate  total_payment_value  average_order_value  average_review_score
        99441             96478                     97.02          16008872.12               160.99                  4.09
